In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import session_processing_helper as helper
import utils

In [2]:
# Configuration
data_dir = '/Users/rebekahzhang/data/behavior_data'
exp = 'exp2'
data_folder = os.path.join(data_dir, exp)
output_folder = os.path.join(data_dir, 'HMM')
os.makedirs(output_folder, exist_ok=True)

In [14]:
# sessions_training = utils.load_session_log(data_folder,'sessions_training_filtered2.csv')
sessions_training = pd.read_csv(os.path.join(data_folder, f'sessions_training_{exp}.csv'))
session_info = sessions_training.loc[sessions_training.dir == '2024-02-02_10-34-46_RZ036'].iloc[0]
events = utils.load_data(utils.generate_events_processed_path(data_folder, session_info))
trials = utils.load_data(utils.generate_trials_path(data_folder, session_info))
trials_data = helper.get_trial_data_df(events.groupby('session_trial_num'))
trials_analyzed = pd.merge(trials, trials_data, on='session_trial_num')

In [ ]:
# trials_filtered = pd.read_csv(os.path.join(data_folder, "trials_training_filtered2.csv"))

# Reward sensitivity

In [ ]:
trials_filtered['delta_tw'] = trials_filtered['time_waited'] - trials_filtered['previous_trial_time_waited']
trials_filtered_not_miss = trials_filtered.loc[trials_filtered['miss_trial'] == False].copy()

In [ ]:
plot_data_clean = trials_filtered_not_miss.dropna(subset=['delta_tw', 'previous_trial_reward', 'group'])

# Get unique groups
groups = plot_data_clean['group'].unique()

# Print statistics
print("=" * 60)
print("DELTA TW STATISTICS BY GROUP AND PREVIOUS TRIAL REWARD")
print("=" * 60)
for group in groups:
    data_group = plot_data_clean[plot_data_clean['group'] == group]
    print(f"\nGroup {group.upper()}:")
    for reward in sorted(data_group['previous_trial_reward'].unique()):
        reward_data = data_group[data_group['previous_trial_reward'] == reward]
        reward_label = "Unrewarded (0)" if reward == 0 else "Rewarded (5)"
        print(f"  {reward_label}:")
        print(f"    Mean: {reward_data['delta_tw'].mean():.4f}")
        print(f"    Std:  {reward_data['delta_tw'].std():.4f}")
        print(f"    N:    {len(reward_data)}")
print("=" * 60)

# Create separate plots for each group
fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 6))
if len(groups) == 1:
    axes = [axes]

# Define colors: blue for unrewarded (0), red for rewarded (5)
colors = {0: 'blue', 5: 'red'}

for idx, group in enumerate(groups):
    data_group = plot_data_clean[plot_data_clean['group'] == group]
    
    # Use violin plot for better visualization of large datasets
    sns.violinplot(data=data_group, x='previous_trial_reward', y='delta_tw', 
                   hue='previous_trial_reward', palette=colors, ax=axes[idx])
    
    axes[idx].set_xlabel('Previous Trial Reward')
    axes[idx].set_ylabel('Delta TW')
    axes[idx].set_ylim(-10, 10)
    axes[idx].set_title(f'Group {group.upper()}: Delta TW by Previous Trial Reward (n={len(data_group)})')
    
    # Customize legend
    handles, labels = axes[idx].get_legend_handles_labels()
    axes[idx].legend(handles, ['Unrewarded (0 ul)', 'Rewarded (5 ul)'], title='Previous Trial')

plt.tight_layout()
plt.savefig(os.path.join(output_folder, 'delta_tw_by_previous_reward.png'))

In [ ]:
# Create quantiles based on PREVIOUS trial's time_waited
trials_quantile = trials_filtered_not_miss.copy()

# Create quantiles of previous_trial_time_waited
trials_quantile['prev_tw_quantile'] = pd.qcut(
    trials_quantile['previous_trial_time_waited'], 
    q=4, 
    labels=['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)'],
    duplicates='drop'
)

print("Previous Trial Time Waited Quantile distribution:")
print(trials_quantile['prev_tw_quantile'].value_counts().sort_index())
print(f"\nTotal rows with quantiles: {trials_quantile['prev_tw_quantile'].notna().sum()}")

In [ ]:
# Analyze all quantiles in one plot per group
plot_data_q = trials_quantile.dropna(subset=['delta_tw', 'previous_trial_reward', 'group', 'prev_tw_quantile'])

# Print condensed statistics
print("="*70)
print("DELTA TW BY QUANTILE, REWARD, AND GROUP")
print("="*70)
for group in ['l', 's']:
    print(f"\nGroup {group.upper()}:")
    data_group = plot_data_q[plot_data_q['group'] == group]
    for quantile in sorted(data_group['prev_tw_quantile'].unique()):
        q_data = data_group[data_group['prev_tw_quantile'] == quantile]
        unrewarded = q_data[q_data['previous_trial_reward'] == 0]['delta_tw'].mean()
        rewarded = q_data[q_data['previous_trial_reward'] == 5]['delta_tw'].mean()
        print(f"  {quantile}: Unrewarded={unrewarded:.3f}, Rewarded={rewarded:.3f}, Diff={rewarded-unrewarded:.3f}")

# Create combined plot: L on left, S on right
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {0: 'blue', 5: 'red'}

for idx, group in enumerate(['l', 's']):
    data_group = plot_data_q[plot_data_q['group'] == group]
    sns.violinplot(data=data_group, x='prev_tw_quantile', y='delta_tw', 
                   hue='previous_trial_reward', palette=colors, ax=axes[idx], split=False)
    axes[idx].set_xlabel('Previous Trial Time Waited Quantile')
    axes[idx].set_ylabel('Delta TW')
    axes[idx].set_ylim(-7.5, 7.5)
    axes[idx].set_title(f'Group {group.upper()} (n={len(data_group)})')
    
    # Customize legend
    handles, labels = axes[idx].get_legend_handles_labels()
    axes[idx].legend(handles, ['Unrewarded (0 ul)', 'Rewarded (5 ul)'], title='Previous Trial Reward')

plt.tight_layout()
plt.savefig(os.path.join(output_folder, 'delta_tw_by_prev_tw_quantile_and_reward.png'))